# Pommerman FFA Baselines

Train and evaluate practical baseline policies for full four-agent Pommerman FFA. The defaults are smoke-test sized; increase `N_EPISODES` for meaningful learning curves.


In [ ]:
from pathlib import Path
import sys


def _find_repo_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for path in (current, *current.parents):
        if (path / "discrete_action_space").exists() and (path / "relevant_papers").exists():
            return path
    raise RuntimeError(f"Could not find repo root from {current}")

ROOT = _find_repo_root()
for path in (ROOT, ROOT / "discrete_action_space", ROOT / "discrete_action_space" / "bimatrix_game"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from discrete_action_space.pommerman_ffa.notebook_utils import (
    display_evaluation_rollout,
    display_training_reward_plots,
    evaluate_policy,
    evaluate_random_reference,
    evaluate_simple_agent_reference,
    policy_from_iql,
    policy_from_ippo,
    train_iql_dqn,
    train_ippo,
)

POMMERMAN_DIR = ROOT / "discrete_action_space" / "pommerman_ffa"

N_EPISODES = 100
MAX_STEPS = 200
EVAL_EPISODES = 20
EVAL_VIDEO_FPS = 4
SEED = 2025
USE_GPU = True
OUTPUT_ROOT = POMMERMAN_DIR / "runs" / "baselines"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


## Random Reference Evaluation

Run fresh evaluation episodes, show the first rollout video if rendering returns frames, then plot one reward boxplot with one box per agent.

In [ ]:
random_eval = evaluate_random_reference(
    n_episodes=EVAL_EPISODES,
    max_steps=MAX_STEPS,
    seed=SEED + 1000,
    output_dir=OUTPUT_ROOT / "random",
)
display_evaluation_rollout(random_eval, fps=EVAL_VIDEO_FPS)


## Built-In SimpleAgent Evaluation

Run fresh evaluation episodes, show the first rollout video if rendering returns frames, then plot one reward boxplot with one box per agent.

In [ ]:
simple_eval = evaluate_simple_agent_reference(
    n_episodes=EVAL_EPISODES,
    max_steps=MAX_STEPS,
    seed=SEED + 2000,
    output_dir=OUTPUT_ROOT / "simple_agent",
)
display_evaluation_rollout(simple_eval, fps=EVAL_VIDEO_FPS)


## Shared-Parameter IQL/DQN Training

In [ ]:
iql_stats = train_iql_dqn(
    n_episodes=N_EPISODES,
    max_steps=MAX_STEPS,
    seed=SEED,
    output_root=OUTPUT_ROOT,
    use_gpu=USE_GPU,
)


## Shared-Parameter IQL/DQN Training Reward Plots

One scatter plot per agent with mean/std, followed by one combined agent reward curve.

In [ ]:
iql_training_figs = display_training_reward_plots(iql_stats)


## Shared-Parameter IQL/DQN Evaluation

Run the trained policy for `EVAL_EPISODES`, show the first rollout video if possible, then plot one boxplot with each agent reward distribution.

In [ ]:
iql_eval = evaluate_policy(
    policy_from_iql(iql_stats["agent"]),
    n_episodes=EVAL_EPISODES,
    max_steps=MAX_STEPS,
    seed=SEED + 3000,
    output_dir=OUTPUT_ROOT / "iql_dqn",
    label="iql_dqn",
)
display_evaluation_rollout(iql_eval, fps=EVAL_VIDEO_FPS)


## Shared-Parameter IPPO/PPO Training

In [ ]:
ippo_stats = train_ippo(
    n_episodes=N_EPISODES,
    max_steps=MAX_STEPS,
    seed=SEED + 10,
    output_root=OUTPUT_ROOT,
    use_gpu=USE_GPU,
)


## Shared-Parameter IPPO/PPO Training Reward Plots

One scatter plot per agent with mean/std, followed by one combined agent reward curve.

In [ ]:
ippo_training_figs = display_training_reward_plots(ippo_stats)


## Shared-Parameter IPPO/PPO Evaluation

Run the trained policy for `EVAL_EPISODES`, show the first rollout video if possible, then plot one boxplot with each agent reward distribution.

In [ ]:
ippo_eval = evaluate_policy(
    policy_from_ippo(ippo_stats["agent"]),
    n_episodes=EVAL_EPISODES,
    max_steps=MAX_STEPS,
    seed=SEED + 4000,
    output_dir=OUTPUT_ROOT / "ippo",
    label="ippo",
)
display_evaluation_rollout(ippo_eval, fps=EVAL_VIDEO_FPS)
